In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from datetime import timedelta

def load_data(file_path):
    raw_data = pd.read_csv(file_path)
    print(f"Data loaded from {file_path}. Shape: {raw_data.shape}")
    print(f"Columns: {raw_data.columns.tolist()}")
    return raw_data

def transform_date(file_path):
    file_raw_date = file_path[10:20]
    print(f"Pre court raw date: {file_raw_date}")
    fecha_archivo = pd.to_datetime(file_raw_date, format='%d.%m.%Y').date()
    fecha_produccion = fecha_archivo + timedelta(days=1)
    print(f"Fecha archivo: {fecha_archivo} -> Fecha produccion esperada (dia siguiente): {fecha_produccion}")
    return fecha_produccion

def compare_precourt_flash(date_precourt, precourt, flash):
    flash['Fecha de factura'] = pd.to_datetime(flash['Fecha de factura'], format='%d/%m/%Y').dt.date
    flash['Date Match'] = flash['Fecha de factura'] == date_precourt
    print(f"Comparison of Pre court date with each invoice date in Flash:")
    print(flash[flash['Date Match'] == True])
    return flash



def main():
    precourt_path = "PRE CORTE 13.02.2026.xlsx - NOTIFICACION.csv"
    flash_path = "FLASH.xlsx - Sheet1.csv"
    precourt = load_data(precourt_path)
    flash = load_data(flash_path)
    date_precourt = transform_date(precourt_path)
    flash_with_comparison = compare_precourt_flash(date_precourt, precourt, flash)

    

main()

Data loaded from PRE CORTE 13.02.2026.xlsx - NOTIFICACION.csv. Shape: (37, 13)
Columns: ['MATERIAL', 'REFERENCIA', 'NECESIDAD BANDEJA', 'NECESIDAD UNIDADES', 'FISICO BANDEJAS', 'FISICO UNIDADES', 'PRODUCIR BANDEJAS', 'PRODUCIR UNIDADES', 'NOTIFICADO', 'PRODUCCION', 'NOTIFICAR', 'PRODUCCION.1', 'NOTIFICAR.1']
Data loaded from FLASH.xlsx - Sheet1.csv. Shape: (35421, 25)
Columns: ['Factura', 'Posicion', 'Cantidad Neta', 'Documento de ventas', 'Centro', 'Oficina de ventas', 'Almacen', 'Zona de ventas', 'Gr Cliente', 'Canal Dist', 'Motivo Dev', 'Fact anulada', 'Cl factura', 'Devolucion', 'Fecha de factura', 'Facturado Real', 'Zona Vtas Dest', 'Gr. Materiales', 'Material', 'Nomb Material', 'Solicitante', 'Nomb Solicitante', 'Destinatario', 'Nomb Destinatario', 'STCD1']
Pre court raw date: 13.02.2026
Fecha archivo: 2026-02-13 -> Fecha produccion esperada (dia siguiente): 2026-02-14
Comparison of Pre court date with each invoice date in Flash:
          Factura  Posicion Cantidad Neta  Documen

In [41]:
flash_check = pd.read_csv("FLASH.xlsx - Sheet1.csv")
flash_check['Fecha de factura'] = pd.to_datetime(flash_check['Fecha de factura'], format='%d/%m/%Y').dt.date

conteo_por_fecha = flash_check['Fecha de factura'].value_counts().sort_index()
print("Conteo de filas por fecha en FLASH (top 10):")
print(conteo_por_fecha.head(10))

from datetime import date
esperado_13 = 1377
esperado_14 = 1325
real_13 = int(conteo_por_fecha.get(date(2026, 2, 13), 0))
real_14 = int(conteo_por_fecha.get(date(2026, 2, 14), 0))

print(f"\nValidacion Fase 0:")
print(f"  13/02/2026 (fecha del nombre del archivo): {real_13} filas  [esperado {esperado_13}]  {'OK' if real_13 == esperado_13 else 'REVISAR'}")
print(f"  14/02/2026 (dia siguiente, el que ahora se compara): {real_14} filas  [esperado {esperado_14}]  {'OK' if real_14 == esperado_14 else 'REVISAR'}")
print(f"\nFix aplicado correctamente: el cruce ahora usa las {real_14} filas del 14/02, no las {real_13} del 13/02.")

Conteo de filas por fecha en FLASH (top 10):
Fecha de factura
2026-02-01     113
2026-02-02    1361
2026-02-03    1543
2026-02-04    1601
2026-02-05    1686
2026-02-06    1540
2026-02-07    1334
2026-02-08      29
2026-02-09    1259
2026-02-10    1652
Name: count, dtype: int64

Validacion Fase 0:
  13/02/2026 (fecha del nombre del archivo): 1377 filas  [esperado 1377]  OK
  14/02/2026 (dia siguiente, el que ahora se compara): 1325 filas  [esperado 1325]  OK

Fix aplicado correctamente: el cruce ahora usa las 1325 filas del 14/02, no las 1377 del 13/02.
